In [66]:
import pandas as pd
import numpy as np

In [67]:
df = pd.read_csv("powerplant_data.csv")

In [68]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [70]:
# AT => Temperature
# V => Vacuum
# AP => Pressure
# RH => Humidity

# PE => Produced Energy

In [71]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [72]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [73]:
X.head()

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20


In [74]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [75]:
X_train

,AT,V,AP,RH
5487,25.24,63.47,1011.30,66.21
3522,26.09,70.40,1007.41,85.37
6916,26.63,73.68,1015.15,85.13
7544,32.06,71.85,1007.90,56.44
7600,28.70,71.64,1007.11,69.85
...,...,...,...,...
5734,26.25,61.02,1011.47,71.22
5191,29.17,64.79,1016.43,61.05
5390,18.00,43.70,1015.40,61.28
860,26.73,68.84,1010.75,66.83


In [76]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [77]:
import torch
import torch.nn as nn

In [78]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [79]:
from torch.utils.data import TensorDataset, DataLoader

In [80]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [81]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

Deep Learninng

In [82]:
# Define our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [83]:
import torch.optim as optim

model = ANN()

#losses, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [84]:
#Train th eANN
train_losses = []
val_losses = []
epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 #total traning loss for 1 epoch

    for xb, yb in train_loader:
        #xb = feature of 1 batch
        #yb = labels of 1 batch
        optimizer.zero_grad()

        outputs = model(xb) # forward prop.. predictes outputs for this batch
        loss = crietrion(outputs, yb) #complete loss
        loss.backward() #back prop.. complete gradients
        optimizer.step() #parameter update

        running_loss += loss.item() #loss is a tensor => python float
    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    #Validation 
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): #no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb) 
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")



epoch 1/100 ==> train loss = 206344.70904947916 & val loss = 204631.140625
epoch 2/100 ==> train loss = 198838.7896484375 & val loss = 188946.515625
epoch 3/100 ==> train loss = 170591.04342447917 & val loss = 148342.46875
epoch 4/100 ==> train loss = 123790.46907552083 & val loss = 100619.5
epoch 5/100 ==> train loss = 83312.78063151041 & val loss = 69651.4140625
epoch 6/100 ==> train loss = 59131.66713867187 & val loss = 50054.4453125
epoch 7/100 ==> train loss = 41106.780436197914 & val loss = 32878.3203125
epoch 8/100 ==> train loss = 25368.332873535157 & val loss = 18813.912109375
epoch 9/100 ==> train loss = 13999.040340169271 & val loss = 10067.8642578125
epoch 10/100 ==> train loss = 7827.764705403646 & val loss = 5832.775390625
epoch 11/100 ==> train loss = 4823.766035970052 & val loss = 3751.468505859375
epoch 12/100 ==> train loss = 3192.622019958496 & val loss = 2586.079833984375
epoch 13/100 ==> train loss = 2249.994907124837 & val loss = 1910.8414306640625
epoch 14/100 ==